In [2]:
import pandas as pd
import arcpy

In [6]:
import os

ENVIRONMENT = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis_recovered.gdb'
streets_network = r'Streets_Network_Drive'
oid_field = "OBJECTID"  # OID field name for streets_network

# polygons (names resolved against ENVIRONMENT workspace)
flood_zones = r'FEMA_Flood_Zone'
high_severity_fire = r'ForestManagementZone_Identity_HighSeverityProbability'

# rasters - full absolute paths
landslide_raster = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\usgs_landslide_risk.tif'

In [7]:
# --- GDB lock diagnostic / fix ---
# Run this cell if AddField fails with ERROR 000229 "Cannot open"
import glob

lock_files = glob.glob(os.path.join(ENVIRONMENT, "*.lock"))
if lock_files:
    print("Stale lock files found (safe to delete if no ArcGIS processes are running):")
    for f in lock_files:
        print(f"  {f}")
else:
    print("No .lock files found.")

# Compact releases any held file GDB connections
try:
    arcpy.management.Compact(ENVIRONMENT)
    print("GDB compacted - any held locks released.")
except Exception as e:
    print(f"Compact failed: {e}")

Stale lock files found (safe to delete if no ArcGIS processes are running):
  F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis_recovered.gdb\Streets_Network_Drive_Landslide.LT-7N3B4W3.61588.17348.sr.lock
  F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis_recovered.gdb\_gdb.LT-7N3B4W3.61588.17348.sr.lock
GDB compacted - any held locks released.


## Polygon intersection flag

Add a binary field (1/0) to a line layer indicating whether each feature intersects a polygon layer.
Polygon layer can be pre-filtered with an optional SQL `where_clause`.

In [8]:
def intersect_flag(line_layer, polygon_layer, flag_field, where_clause=None):
    """
    Adds flag_field (SHORT) to line_layer: 1 if the line intersects the
    (optionally filtered) polygon_layer, 0 otherwise.

    Spatial work uses temp feature layers (read-only on the original).
    Values are written back via UpdateCursor - no schema lock needed once
    the field exists. If the field is missing, AddField is attempted; if
    that fails due to a lock, add a SHORT field named flag_field manually
    in ArcGIS Pro then re-run - the cursor write will work.
    """
    for tmp in ("poly_lyr_tmp", "line_lyr_tmp"):
        if arcpy.Exists(tmp):
            arcpy.management.Delete(tmp)

    # Add field only if missing - UpdateCursor on an existing field does not need a schema lock
    if flag_field not in [f.name for f in arcpy.ListFields(line_layer)]:
        try:
            arcpy.management.AddField(line_layer, flag_field, "SHORT")
        except arcpy.ExecuteError:
            raise RuntimeError(
                f"Cannot add field '{flag_field}' to '{line_layer}' - schema is locked.\n"
                f"Add a SHORT field named '{flag_field}' manually in ArcGIS Pro, then re-run."
            )

    try:
        poly_lyr = arcpy.management.MakeFeatureLayer(
            polygon_layer, "poly_lyr_tmp", where_clause
        ).getOutput(0)
        line_lyr = arcpy.management.MakeFeatureLayer(line_layer, "line_lyr_tmp").getOutput(0)
        arcpy.management.SelectLayerByLocation(line_lyr, "INTERSECT", poly_lyr)

        intersecting_oids = {row[0] for row in arcpy.da.SearchCursor(line_lyr, ["OID@"])}

        with arcpy.da.UpdateCursor(line_layer, ["OID@", flag_field]) as cur:
            for row in cur:
                row[1] = 1 if row[0] in intersecting_oids else 0
                cur.updateRow(row)
    finally:
        for tmp in ("poly_lyr_tmp", "line_lyr_tmp"):
            if arcpy.Exists(tmp):
                arcpy.management.Delete(tmp)

    print(f"Done: {len(intersecting_oids)} intersecting features flagged in '{flag_field}'.")


# --- Example usage ---
arcpy.env.workspace = ENVIRONMENT

intersect_flag(streets_network, flood_zones, "in_flood_zone")
intersect_flag(streets_network, high_severity_fire, "high_sev_fire")

Done: 696 intersecting features flagged in 'in_flood_zone'.
Done: 18285 intersecting features flagged in 'high_sev_fire'.


## Raster zonal statistics

Compute zonal statistics of a raster for each line feature and join the result back as a new field.
Uses `ZonalStatisticsAsTable` (Spatial Analyst) + `JoinField`.

`statistics_type` accepts any single ZonalStatisticsAsTable keyword: `MEAN`, `MIN`, `MAX`, `SUM`,
`STD`, `MEDIAN`, `RANGE`, `COUNT`. Do not pass `ALL`.

The `zone_field` defaults to the OID field, which is unique per feature and requires no prep.
Supply a different field if you need to aggregate multiple segments to the same zone.

In [11]:
SCRATCH_GDB = r"C:\GIS\Scratch.gdb"

# ZonalStatisticsAsTable input keyword -> output table column name
_ZONAL_STAT_COL = {
    "MEAN": "MEAN", "MINIMUM": "MIN", "MAXIMUM": "MAX",
    "SUM": "SUM", "STD": "STD", "MEDIAN": "MEDIAN",
    "RANGE": "RANGE", "VARIETY": "VARIETY", "MAJORITY": "MAJORITY",
    "MINORITY": "MINORITY", "COUNT": "COUNT",
}


def raster_zonal_stats(
    line_layer,
    raster,
    out_field,
    statistics_type="MEAN",
    zone_field="OBJECTID",
    out_workspace=None,
):
    """
    Calculates zonal statistics of raster for each line feature and writes
    the stat back to line_layer as out_field.

    All schema work is done on a scratch copy in SCRATCH_GDB, so the
    original feature class is never schema-locked by this function.

    If out_field does not exist yet on the original, AddField is attempted.
    If that fails due to a lock, add a DOUBLE field named out_field manually
    in ArcGIS Pro, then re-run - the cursor write will work.

    zone_field: unique integer field in line_layer (default "OBJECTID").
    statistics_type: MEAN | MINIMUM | MAXIMUM | SUM | STD | MEDIAN |
                     RANGE | VARIETY | MAJORITY | MINORITY | COUNT
    """
    raster = str(raster)
    if os.path.basename(raster).lower().startswith("band_"):
        raster = os.path.dirname(raster)
    if not arcpy.Exists(raster):
        raise ValueError(f"Raster not found: '{raster}'")

    stat_key = statistics_type.upper()
    if stat_key not in _ZONAL_STAT_COL:
        raise ValueError(f"Invalid statistics_type '{statistics_type}'. Valid: {list(_ZONAL_STAT_COL)}")
    stat_col = _ZONAL_STAT_COL[stat_key]

    scratch_gdb = out_workspace or SCRATCH_GDB
    if not arcpy.Exists(scratch_gdb):
        arcpy.management.CreateFileGDB(os.path.dirname(scratch_gdb), os.path.basename(scratch_gdb))

    scratch_fc = os.path.join(scratch_gdb, "zone_fc_tmp")
    out_table  = os.path.join(scratch_gdb, "zonal_stats_tmp")

    for tmp in (scratch_fc, out_table):
        if arcpy.Exists(tmp):
            arcpy.management.Delete(tmp)

    try:
        arcpy.management.CopyFeatures(line_layer, scratch_fc)
        if not arcpy.Exists(scratch_fc):
            raise RuntimeError(
                f"CopyFeatures did not create '{scratch_fc}'.\n"
                "The source feature class may be corrupted or unreadable."
            )

        tmp_id = "_zone_id_tmp"
        arcpy.management.AddField(scratch_fc, tmp_id, "LONG")
        with arcpy.da.UpdateCursor(scratch_fc, [zone_field, tmp_id]) as cur:
            for row in cur:
                row[1] = row[0]
                cur.updateRow(row)

        arcpy.sa.ZonalStatisticsAsTable(
            scratch_fc, tmp_id, raster, out_table,
            ignore_nodata="DATA",
            statistics_type=stat_key,
        )

        lookup = {int(r[0]): r[1] for r in arcpy.da.SearchCursor(out_table, [tmp_id, stat_col])}

        if out_field not in [f.name for f in arcpy.ListFields(line_layer)]:
            try:
                arcpy.management.AddField(line_layer, out_field, "DOUBLE")
            except arcpy.ExecuteError:
                raise RuntimeError(
                    f"Cannot add field '{out_field}' to '{line_layer}' - schema is locked.\n"
                    f"Add a DOUBLE field named '{out_field}' manually in ArcGIS Pro, then re-run."
                )

        with arcpy.da.UpdateCursor(line_layer, [zone_field, out_field]) as cur:
            for row in cur:
                row[1] = lookup.get(int(row[0]))
                cur.updateRow(row)

    finally:
        for tmp in (scratch_fc, out_table):
            if arcpy.Exists(tmp):
                arcpy.management.Delete(tmp)

    print(f"Done: '{out_field}' ({stat_key}) written to '{line_layer}'.")


# --- Example usage ---
arcpy.env.workspace = ENVIRONMENT

raster_zonal_stats(streets_network, landslide_raster, "landslide_mean", statistics_type="MEAN",    zone_field=oid_field)
raster_zonal_stats(streets_network, landslide_raster, "landslide_max",  statistics_type="MAXIMUM", zone_field=oid_field)

Done: 'landslide_mean' (MEAN) written to 'Streets_Network_Drive'.
Done: 'landslide_max' (MAXIMUM) written to 'Streets_Network_Drive'.
